# Momentum

**Capítulo 2 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_optimization/momentum.ipynb` · [Lección original](https://d2l.ai/chapter_optimization/momentum.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Momentum
<a id="sec_momentum"></a>

En [Referencia sec_sgd](https://d2l.ai/chapter_optimization/sgd.html#sec-sgd) revisamos lo que sucede cuando se realiza descenso por gradiente estocástico, es decir, cuando se realiza la optimización donde sólo hay disponible una variante ruidosa del gradiente. En particular, nos dimos cuenta de que para los gradientes ruidosos tenemos que ser extremadamente cautelosos a la hora de elegir la tasa de aprendizaje frente al ruido. Si lo disminuimos demasiado rápido, se para la convergencia. Si somos demasiado indulgentes, no convergimos a una solución lo suficientemente buena, ya que el ruido sigue alejandonos de la optimalidad.

## Básicos
En esta sección, exploraremos algoritmos de optimización más eficaces, especialmente para ciertos tipos de problemas de optimización que son comunes en la práctica.

### Medias móviles
La sección anterior nos vio discutir minibatch SGD como un medio para acelerar el cálculo. También tuvo el agradable efecto secundario que el promedio de gradientes redujo la cantidad de varianza. El descenso por gradiente minibatch estocástico se puede calcular por:

$$\mathbf{g}_{t, t-1} = \partial_{\mathbf{w}} \frac{1}{|\mathcal{B}_t|} \sum_{i \in \mathcal{B}_t} f(\mathbf{x}_{i}, \mathbf{w}_{t-1}) = \frac{1}{|\mathcal{B}_t|} \sum_{i \in \mathcal{B}_t} \mathbf{h}_{i, t-1}.
$$

Para mantener la notación simple, aquí usamos $\mathbf{h}_{i, t-1} = \partial_{\mathbf{w}} f(\mathbf{x}_i, \mathbf{w}_{t-1})$ como el descenso por gradiente estocástico para la muestra $i$ utilizando los pesos actualizados en el tiempo $t-1$. Sería bueno si pudiéramos beneficiarnos del efecto de reducción de varianza incluso más allá de los gradientes promedio en un minibatch. Una opción para lograr esta tarea es reemplazar el cálculo de gradiente por un "media de marea":

$$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \mathbf{g}_{t, t-1}$$

para algunos $\beta \in (0, 1)$. Esto reemplaza efectivamente el gradiente instantáneo por uno que se ha promediado sobre múltiples gradientes *pasados*. $\mathbf{v}$ se llama *velocidad*. Se acumula gradientes pasados similares a cómo una bola pesada rodando hacia abajo el paisaje de la función objetiva se integra sobre fuerzas pasadas. Para ver lo que está sucediendo en más detalle vamos a expandir $\mathbf{v}_t$ recursivamente en

$$\begin{aligned}
\mathbf{v}_t = \beta^2 \mathbf{v}_{t-2} + \beta \mathbf{g}_{t-1, t-2} + \mathbf{g}_{t, t-1}
= \ldots, = \sum_{\tau = 0}^{t-1} \beta^{\tau} \mathbf{g}_{t-\tau, t-\tau-1}.
\end{aligned}$$

El $\beta$ grande es un promedio de largo alcance, mientras que el $\beta$ pequeño es sólo una ligera corrección relativa a un método de gradiente. El nuevo reemplazo de gradiente ya no apunta en la dirección de descenso más pronunciado en una instancia particular más tiempo, sino en la dirección de un promedio ponderado de gradientes pasados. Esto nos permite realizar la mayoría de los beneficios de promediar sobre un lote sin el costo de calcular realmente los gradientes en él. Vamos a volver a este procedimiento de promedio en más detalle más adelante.

El razonamiento anterior constituyó la base de lo que ahora se conoce como métodos de gradiente acelerados, tales como gradientes con impulso. Gozan del beneficio adicional de ser mucho más efectivos en los casos en que el problema de optimización está mal condicionado (es decir, donde hay algunas direcciones en las que el progreso es mucho más lento que en otras, parecido a un cañón estrecho). Además, nos permiten promediar sobre gradientes posteriores para obtener direcciones de descenso más estables. De hecho, el aspecto de aceleración incluso para problemas convexos sin ruido es una de las razones clave por las que el impulso funciona y por qué funciona tan bien.

Como cabría esperar, debido a su impulso de eficacia es un tema bien estudiado en la optimización para el aprendizaje profundo y más allá. Véase, por ejemplo, el hermoso [expository article](https://distill.pub/2017/momentum/) de [Goh.2017](https://d2l.ai/chapter_references/zreferences.html) para un análisis en profundidad y la animación interactiva. Fue propuesto por [Polyak.1964](https://d2l.ai/chapter_references/zreferences.html). [Nesterov.2018](https://d2l.ai/chapter_references/zreferences.html) tiene una discusión teórica detallada en el contexto de la optimización convexa. Momentum en el aprendizaje profundo se ha sabido que es beneficioso durante mucho tiempo. Véase, por ejemplo, la discusión de [Sutskever.Martens.Dahl.ea.2013](https://d2l.ai/chapter_references/zreferences.html) para detalles.

### Un problema mal condicionado
Para obtener una mejor comprensión de las propiedades geométricas del método de impulso revisitamos el descenso por gradiente, aunque con una función objetiva significativamente menos agradable. Recordemos que en [Referencia sec_gd](https://d2l.ai/chapter_optimization/gd.html#sec-gd) usamos $f(\mathbf{x}) = x_1^2 + 2 x_2^2$, es decir, un objetivo elipsoide moderadamente distorsionado. Distorsionamos esta función al estirarla en la dirección $x_1$ a través de

$$f(\mathbf{x}) = 0.1 x_1^2 + 2 x_2^2.$$

Como antes $f$ tiene su mínimo en $(0, 0)$. Esta función es *muy* plana en la dirección de $x_1$. Veamos qué sucede cuando realizamos descenso por gradiente como antes en esta nueva función. Escogemos una tasa de aprendizaje de $0.4$.


In [ ]:
%matplotlib inline
import torch
from laboratorio import d2l

eta = 0.4
def f_2d(x1, x2):
    return 0.1 * x1 ** 2 + 2 * x2 ** 2
def gd_2d(x1, x2, s1, s2):
    return (x1 - eta * 0.2 * x1, x2 - eta * 4 * x2, 0, 0)

d2l.show_trace_2d(f_2d, d2l.train_2d(gd_2d))

Por construcción, el gradiente en la dirección $x_2$ es *mucho* más alto y cambia mucho más rápidamente que en la dirección horizontal $x_1$. Por lo tanto, nos quedamos atascados entre dos opciones indeseables: si escogemos una pequeña tasa de aprendizaje nos aseguramos de que la solución no diverja en la dirección $x_2$ pero estamos cargados con una lenta convergencia en la dirección $x_1$. Por el contrario, con una gran tasa de aprendizaje progresamos rápidamente en la dirección $x_1$ pero divergimos en $x_2$. El ejemplo siguiente ilustra lo que sucede incluso después de un ligero aumento en la tasa de aprendizaje de $0.4$ a $0.6$. Convergencia en la dirección $x_1$, mejora pero la calidad general de la solución es mucho peor.


In [ ]:
eta = 0.6
d2l.show_trace_2d(f_2d, d2l.train_2d(gd_2d))

### El Método Momentum
El método momentum nos permite resolver el problema de descenso por gradiente descrito anteriormente. Mirando el rastro de optimización arriba podríamos intuir que los gradientes promedios sobre el pasado funcionarían bien. Después de todo, en la dirección $x_1$ esto sumará gradientes bien alineados, aumentando así la distancia que cubrimos con cada paso. A la inversa, en la dirección $x_2$ donde oscilan los gradientes, un gradiente agregado reducirá el tamaño del paso debido a oscilaciones que se cancelan mutuamente. Usando $\mathbf{v}_t$ en lugar del gradiente $\mathbf{g}_t$ produce las siguientes ecuaciones de actualización:

$$
\begin{aligned}
\mathbf{v}_t &\leftarrow \beta \mathbf{v}_{t-1} + \mathbf{g}_{t, t-1}, \\
\mathbf{x}_t &\leftarrow \mathbf{x}_{t-1} - \eta_t \mathbf{v}_t.
\end{aligned}
$$

Tenga en cuenta que para $\beta = 0$ recuperamos el descenso por gradiente regular. Antes de profundizar en las propiedades matemáticas vamos a echar un vistazo rápido a cómo se comporta el algoritmo en la práctica.


In [ ]:
def momentum_2d(x1, x2, v1, v2):
    v1 = beta * v1 + 0.2 * x1
    v2 = beta * v2 + 4 * x2
    return x1 - eta * v1, x2 - eta * v2, v1, v2

eta, beta = 0.6, 0.5
d2l.show_trace_2d(f_2d, d2l.train_2d(momentum_2d))

Como podemos ver, incluso con la misma tasa de aprendizaje que usamos antes, el impulso aún converge bien. Veamos qué sucede cuando decrecemos el parámetro de impulso. Reducirlo a la mitad a $\beta = 0.25$ conduce a una trayectoria que apenas converge en absoluto. Sin embargo, es mucho mejor que sin impulso (cuando la solución diverge).


In [ ]:
eta, beta = 0.6, 0.25
d2l.show_trace_2d(f_2d, d2l.train_2d(momentum_2d))

El único cambio es que en ese caso reemplazamos los gradientes $\mathbf{g}_{t, t-1}$ por $\mathbf{g}_t$. Por último, para comodidad inicializamos $\mathbf{v}_0 = 0$ en el momento $t=0$. Veamos lo que el promedio de fugas realmente hace a las actualizaciones.

### Peso efectivo de la muestra
Recuerde que $\mathbf{v}_t = \sum_{\tau = 0}^{t-1} \beta^{\tau} \mathbf{g}_{t-\tau, t-\tau-1}$. En el límite los términos suman hasta $\sum_{\tau=0}^\infty \beta^\tau = \frac{1}{1-\beta}$. En otras palabras, en lugar de tomar un paso del tamaño $\eta$ en descenso por gradiente o descenso por gradiente estocástico tomamos un paso del tamaño $\frac{\eta}{1-\beta}$ mientras que al mismo tiempo, tratando con una dirección de descenso potencialmente mucho mejor comportada. Estos son dos beneficios en uno. Para ilustrar cómo se comporta la ponderación para diferentes opciones de $\beta$ considere el diagrama a continuación.


In [ ]:
d2l.set_figsize()
betas = [0.95, 0.9, 0.6, 0]
for beta in betas:
    x = torch.arange(40).detach().numpy()
    d2l.plt.plot(x, beta ** x, label=f'beta = {beta:.2f}')
d2l.plt.xlabel('time')
d2l.plt.legend();

## Experimentos prácticos
Veamos cómo funciona el impulso en la práctica, es decir, cuando se utiliza en el contexto de un optimizador adecuado. Para esto necesitamos una implementación algo más escalable.

### Implementación desde cero
En comparación con (minibatch) descenso por gradiente estocástico el método de impulso necesita mantener un conjunto de variables auxiliares, es decir, velocidad. Tiene la misma forma que los gradientes (y variables del problema de optimización). En la implementación a continuación llamamos a estas variables `states`.


In [ ]:
def init_momentum_states(feature_dim):
    v_w = torch.zeros((feature_dim, 1))
    v_b = torch.zeros(1)
    return (v_w, v_b)

### Nota docente de Hespérides

Sigue tres objetos diferentes: el valor de la pérdida, su gradiente y la actualización que calcula el optimizador. Comprueba formas y reinicia los gradientes antes de cada paso. En el explorador se mantienen función y punto inicial para comparar trayectorias; una misma tasa no significa el mismo desplazamiento efectivo para todos los métodos.

Vínculo con los apuntes: sesión 2, «Momentum».


In [ ]:
def sgd_momentum(params, states, hyperparams):
    for p, v in zip(params, states):
        with torch.no_grad():
            v[:] = hyperparams['momentum'] * v + p.grad
            p[:] -= hyperparams['lr'] * v
        p.grad.data.zero_()

Veamos cómo funciona esto en la práctica.


In [ ]:
def train_momentum(lr, momentum, num_epochs=2):
    d2l.train_ch11(sgd_momentum, init_momentum_states(feature_dim),
                   {'lr': lr, 'momentum': momentum}, data_iter,
                   feature_dim, num_epochs)

data_iter, feature_dim = d2l.get_data_ch11(batch_size=10)
train_momentum(0.02, 0.5)

Cuando aumentamos el hiperparametro de impulso `momentum` a 0.9, equivale a un tamaño de muestra efectiva significativamente mayor de $\frac{1}{1 - 0.9} = 10$. Reducimos ligeramente la tasa de aprendizaje a $0.01$ para mantener los asuntos bajo control.


In [ ]:
train_momentum(0.01, 0.9)

La reducción de la tasa de aprendizaje además aborda cualquier problema de optimización no suave problemas. Establecerlo a $0.005$ produce buenas propiedades de convergencia.


In [ ]:
train_momentum(0.005, 0.9)

### Implementación concisa
Hay muy poco que hacer en Gluon, ya que el solucionador estándar `sgd` ya tenía un impulso incorporado. El ajuste de parámetros coincidentes produce una trayectoria muy similar.


In [ ]:
trainer = torch.optim.SGD
d2l.train_concise_ch11(trainer, {'lr': 0.005, 'momentum': 0.9}, data_iter)

## Análisis teórico
Hasta ahora el ejemplo 2D de $f(x) = 0.1 x_1^2 + 2 x_2^2$ parecía bastante ingenioso. Ahora veremos que esto es realmente bastante representativo de los tipos de problema que uno podría encontrar, al menos en el caso de minimizar las funciones objetivas cuadráticas convexas.

### Funciones cuadráticas convexas
Considere la función

$$h(\mathbf{x}) = \frac{1}{2} \mathbf{x}^\top \mathbf{Q} \mathbf{x} + \mathbf{x}^\top \mathbf{c} + b.$$

Esta es una función cuadrática general. Para matrices definidas positivas $\mathbf{Q} \succ 0$, es decir, para matrices con valores propios positivos esto tiene un minimizador en $\mathbf{x}^* = -\mathbf{Q}^{-1} \mathbf{c}$ con valor mínimo $b - \frac{1}{2} \mathbf{c}^\top \mathbf{Q}^{-1} \mathbf{c}$. Por lo tanto podemos reescribir $h$ como

$$h(\mathbf{x}) = \frac{1}{2} (\mathbf{x} - \mathbf{Q}^{-1} \mathbf{c})^\top \mathbf{Q} (\mathbf{x} - \mathbf{Q}^{-1} \mathbf{c}) + b - \frac{1}{2} \mathbf{c}^\top \mathbf{Q}^{-1} \mathbf{c}.$$

El gradiente es dado por $\partial_{\mathbf{x}} h(\mathbf{x}) = \mathbf{Q} (\mathbf{x} - \mathbf{Q}^{-1} \mathbf{c})$. Es decir, es dado por la distancia entre $\mathbf{x}$ y el minimizador, multiplicado por $\mathbf{Q}$. En consecuencia, también la velocidad es una combinación lineal de términos $\mathbf{Q} (\mathbf{x}_t - \mathbf{Q}^{-1} \mathbf{c})$.

Dado que $\mathbf{Q}$ es positivo definido puede ser descompuesto en su sistema propio vía $\mathbf{Q} = \mathbf{O}^\top \boldsymbol{\Lambda} \mathbf{O}$ para una matriz ortogonal (rotación) $\mathbf{O}$ y una matriz diagonal $\boldsymbol{\Lambda}$ de valores propios positivos. Esto nos permite realizar un cambio de variables de $\mathbf{x}$ a $\mathbf{z} \stackrel{\textrm{def}}{=} \mathbf{O} (\mathbf{x} - \mathbf{Q}^{-1} \mathbf{c})$ para obtener una expresión mucho simplificada:

$$h(\mathbf{z}) = \frac{1}{2} \mathbf{z}^\top \boldsymbol{\Lambda} \mathbf{z} + b'.$$

Aquí $b' = b - \frac{1}{2} \mathbf{c}^\top \mathbf{Q}^{-1} \mathbf{c}$. Puesto que $\mathbf{O}$ es sólo una matriz ortogonal esto no perturba los gradientes de una manera significativa. Expresado en términos de descenso por gradiente $\mathbf{z}$ se convierte en

$$\mathbf{z}_t = \mathbf{z}_{t-1} - \boldsymbol{\Lambda} \mathbf{z}_{t-1} = (\mathbf{I} - \boldsymbol{\Lambda}) \mathbf{z}_{t-1}.$$

El hecho importante en esta expresión es que el descenso por gradiente *no se mezcla* entre diferentes espacios propios. Es decir, cuando se expresa en términos del sistema propio de $\mathbf{Q}$ el problema de optimización procede de una manera coordinada. Esto también se mantiene para

$$\begin{aligned}
\mathbf{v}_t & = \beta \mathbf{v}_{t-1} + \boldsymbol{\Lambda} \mathbf{z}_{t-1} \\
\mathbf{z}_t & = \mathbf{z}_{t-1} - \eta \left(\beta \mathbf{v}_{t-1} + \boldsymbol{\Lambda} \mathbf{z}_{t-1}\right) \\
    & = (\mathbf{I} - \eta \boldsymbol{\Lambda}) \mathbf{z}_{t-1} - \eta \beta \mathbf{v}_{t-1}.
\end{aligned}$$

Al hacer esto acabamos de probar el siguiente teorema: descenso por gradiente con y sin impulso para una función cuadrática convexa se descompone en optimización de coordenadas en la dirección de los autovectores de la matriz cuadrática.

### Funciones escalares
Dado el resultado anterior vamos a ver lo que sucede cuando minimizamos la función $f(x) = \frac{\lambda}{2} x^2$.

$$x_{t+1} = x_t - \eta \lambda x_t = (1 - \eta \lambda) x_t.$$

Siempre que $|1 - \eta \lambda| < 1$ esta optimización converge a un ritmo exponencial ya que después de los pasos $t$ tenemos $x_t = (1 - \eta \lambda)^t x_0$. Esto muestra cómo la tasa de convergencia mejora inicialmente a medida que aumentamos la tasa de aprendizaje $\eta$ hasta $\eta \lambda = 1$. Más allá de eso las cosas divergen y para $\eta \lambda > 2$ el problema de optimización diverge.


In [ ]:
lambdas = [0.1, 1, 10, 19]
eta = 0.1
d2l.set_figsize((6, 4))
for lam in lambdas:
    t = torch.arange(20).detach().numpy()
    d2l.plt.plot(t, (1 - eta * lam) ** t, label=f'lambda = {lam:.2f}')
d2l.plt.xlabel('time')
d2l.plt.legend();

Para analizar la convergencia en el caso del momentum empezamos reescribiendo las ecuaciones de actualización en términos de dos escalares: uno para $x$ y otro para la velocidad $v$. Esto produce:

$$
\begin{bmatrix} v_{t+1} \\ x_{t+1} \end{bmatrix} =
\begin{bmatrix} \beta & \lambda \\ -\eta \beta & (1 - \eta \lambda) \end{bmatrix}
\begin{bmatrix} v_{t} \\ x_{t} \end{bmatrix} = \mathbf{R}(\beta, \eta, \lambda) \begin{bmatrix} v_{t} \\ x_{t} \end{bmatrix}.
$$

Utilizamos $\mathbf{R}$ para denotar el comportamiento de convergencia que rige $2 \times 2$. Después de los pasos $t$ la elección inicial $[v_0, x_0]$ se convierte en $\mathbf{R}(\beta, \eta, \lambda)^t [v_0, x_0]$. Por lo tanto, depende de los valores propios de $\mathbf{R}$ para determinar la velocidad de convergencia. Vea el [Distill post](https://distill.pub/2017/momentum/) de [Goh.2017](https://d2l.ai/chapter_references/zreferences.html) para una gran animación y [Flammarion.Bach.2015](https://d2l.ai/chapter_references/zreferences.html) para un análisis detallado. Uno puede mostrar que la velocidad $0 < \eta \lambda < 2 + 2 \beta$ converge. Este es un rango mayor de parámetros factibles en comparación con $0 < \eta \lambda < 2$ para descenso por gradiente. También sugiere que en general son deseables valores grandes de $\beta$. Más detalles requieren una cantidad justa de detalle técnico y sugerimos que el lector interesado consulte las publicaciones originales.

## Resumen
* El momentum reemplaza los gradientes con un promedio de fugas sobre los gradientes pasados, lo que acelera significativamente la convergencia.
* Es deseable tanto para descenso por gradiente sin ruido como para descenso por gradiente estocástico (ruido).
* El momentum evita el estancamiento del proceso de optimización que es mucho más probable que ocurra para el descenso por gradiente estocástico.
* El número efectivo de gradientes es dado por $\frac{1}{1-\beta}$ debido a la ponderación exponencial de los datos pasados.
* En el caso de problemas cuadráticos convexos esto puede ser analizado explícitamente en detalle.
* La implementación es bastante sencilla, pero requiere que almacenemos un vector de estado adicional (velocidad $\mathbf{v}$).

## Ejercicios
1. Utilice otras combinaciones de hiperparametros de impulso y tasas de aprendizaje y observe y analice los diferentes resultados experimentales.
1. Prueba el descenso por gradiente y el impulso para un problema cuadrático donde tienes múltiples valores propios, es decir, $f(x) = \frac{1}{2} \sum_i \lambda_i x_i^2$, por ejemplo, $\lambda_i = 2^{-i}$. Dibuja cómo disminuyen los valores de $x$ para la inicialización $x_i = 1$.
1. Obtén el valor mínimo y el minimizador de $h(\mathbf{x}) = \frac{1}{2} \mathbf{x}^\top \mathbf{Q} \mathbf{x} + \mathbf{x}^\top \mathbf{c} + b$.
1. ¿Qué cambia cuando realizamos descenso por gradiente estocástico con impulso? ¿Qué sucede cuando usamos descenso por gradiente estocástico minibatch con impulso? ¿Experimentar con los parámetros?


[Debate del original](https://discuss.d2l.ai/t/1070)
